# Tutorial 2_1: 2D Panel under various boundary conditions

In this tutorial, we will consider our 2D panel subject to various boundary conditions and try to understand the effect they can have on the results.

# Standard imports

In [ ]:
try:
    from firedrake import *
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh" -O "/tmp/firedrake-install.sh" && bash "/tmp/firedrake-install.sh"
    from firedrake import *

import numpy as np
import matplotlib.pyplot as plt

# Mesh Definition

In [ ]:
Lx = 0.5
Ly = 0.1
nx = 250
ny = 50

mesh = RectangleMesh(nx, ny, Lx, Ly, quadrilateral=True)

V = FunctionSpace(mesh, 'CG', 1)
W = VectorFunctionSpace(mesh, 'CG', 1)
DG0 = FunctionSpace(mesh, 'DG', 0)

# Material Properties and Problem Definition

In this case we will consider a through-thickness temperature distribution, similar to a aerodynamic panel. First let's consider the roller conditions we have used previously and allow expansion in the y direction.

In [ ]:
alpha = Constant(1.2e-5)
E = Constant(210e9)
nu = Constant(0.3)
T_ref = Constant(0)

mu = E / (2 * (1 + nu))
lmbda_ps = E * nu / (1 - nu**2) 

T = Function(V, name='Temperature [C]')
u = Function(W, name='Displacement [m]')
von_mises_s = Function(DG0, name='von Mises Stress [Pa]')

x, y = SpatialCoordinate(mesh)
T_bottom = Constant(0.0)
T_top = Constant(200.0)
T_expr = T_bottom + (T_top - T_bottom) * (y / Ly)

T.interpolate(T_expr)

bc_mech = [DirichletBC(W.sub(0), Constant(0), 1),
           DirichletBC(W.sub(0), Constant(0), 2)]

y_translation = Function(W).interpolate(as_vector([0.0, 1.0]))
nullspace = VectorSpaceBasis([y_translation])
nullspace.orthonormalize()

# Thermoelastic Solver Setup

In [ ]:
u_mech = TrialFunction(W)
v_mech = TestFunction(W)

def epsilon(u):
    return 0.5 * (grad(u) + grad(u).T)

def sigma(u, T):
    thermal_strain = (2 * lmbda_ps + 2 * mu) * alpha * (T - T_ref)
    return lmbda_ps * div(u) * Identity(2) + 2*mu * epsilon(u) - thermal_strain * Identity(2)
    
mech_form = inner(sigma(u_mech, T), epsilon(v_mech)) * dx

a_mech = lhs(mech_form)
L_mech = rhs(mech_form)

mech_prob = LinearVariationalProblem(a_mech, L_mech, u, bcs=bc_mech, constant_jacobian=True)
mech_solver = LinearVariationalSolver(mech_prob, nullspace=nullspace)

# Solution

As expected, since the material can expand freely in the y-direction and the temperature distribution is also only varying in the y-direction, we recover our 1D result graded in the y-direction.

In [ ]:
mech_solver.solve()

s = sigma(u, T)
s_xx, s_yy = s[0,0], s[1,1]
s_xy = s[0,1]
von_mises_s.project(sqrt(s_xx**2 + s_yy**2 - s_xx*s_yy + 3*s_xy**2)/1e6)

fig, axes = plt.subplots(2, 1, figsize=(10, 10))

cont0 = tricontourf(T, axes=axes[0], cmap='inferno')
fig.colorbar(cont0, ax=axes[0], label='Temperature [C]')
axes[0].set_aspect('equal')
axes[0].set_title("Temperature")

cont1 = tricontourf(von_mises_s, axes=axes[1], levels=100, cmap='viridis')
fig.colorbar(cont1, ax=axes[1], label='von Mises Stress [MPa]')
axes[1].set_aspect('equal')
axes[1].set_title("Thermal Stress")

plt.tight_layout()
plt.show()

# Fully fixed ends

Now let's consider each end fixed in the y direction too.

In [ ]:
bc_mech_clamped = [DirichletBC(W, Constant((0.0, 0.0)), 1),
                   DirichletBC(W, Constant((0.0, 0.0)), 2)]

u_mech = TrialFunction(W)
v_mech = TestFunction(W)

def epsilon(u):
    return 0.5 * (grad(u) + grad(u).T)

def sigma(u, T):
    thermal_strain = (2 * lmbda_ps + 2 * mu) * alpha * (T - T_ref)
    return lmbda_ps * div(u) * Identity(2) + 2*mu * epsilon(u) - thermal_strain * Identity(2)

mech_form_clamped = inner(sigma(u_mech, T), epsilon(v_mech)) * dx
a_mech_clamped = lhs(mech_form_clamped)
L_mech_clamped = rhs(mech_form_clamped)

mech_prob_clamped = LinearVariationalProblem(a_mech_clamped, L_mech_clamped, u, bcs=bc_mech_clamped, constant_jacobian=True)
mech_solver_clamped = LinearVariationalSolver(mech_prob_clamped)

In [ ]:
mech_solver_clamped.solve()

s = sigma(u, T)
s_xx, s_yy = s[0,0], s[1,1]
s_xy = s[0,1]
von_mises_s.project(sqrt(s_xx**2 + s_yy**2 - s_xx*s_yy + 3*s_xy**2)/1e6)

fig, ax = plt.subplots(figsize=(10, 4))

capped_levels = np.linspace(0, 700, 100)
cont = tricontourf(von_mises_s, axes=ax, levels=capped_levels, cmap='viridis')
fig.colorbar(cont, ax=ax, label='von Mises Stress [MPa]')
ax.set_aspect('equal')
ax.set_title("Thermal Stress")

plt.tight_layout()
plt.show()

Here, the model is very obviously over constrained. Generally, fixed boundary conditions are highly incompatible with thermal expansion, as the thermal strain part of the total strain is directly driven by the temperature, leading to extreme stress hot spots.

# Elastic Support (Robin Boundary Conditions)

Finally, let's consider boundaries that are neither completely free nor fully rigid. We can model compliance using **Robin boundary conditions** (e.g., an elastic foundation or spring support). 

Instead of forcing the displacement to zero via a Dirichlet condition, we define a traction vector on the boundary $\Gamma_R$ that opposes the displacement proportionally:
$$\sigma \mathbf{n} = -k \mathbf{u} \quad \text{on } \Gamma_R$$

Where $k$ is the stiffness modulus of the surrounding material.

Applying the principle of virtual work, the boundary integral becomes:
$$\int_{\Gamma_R} (\sigma \mathbf{n}) \cdot \mathbf{v} \, ds = \int_{\Gamma_R} -k \mathbf{u} \cdot \mathbf{v} \, ds$$

Moving this term to the left-hand side of our variational equation yields the updated weak form:
$$\int_{\Omega} \sigma(\mathbf{u}, T) : \epsilon(\mathbf{v}) \, dx + \int_{\Gamma_R} k \mathbf{u} \cdot \mathbf{v} \, ds = 0$$

In [ ]:
k_spring = Constant(5e10)

mech_form_robin = inner(sigma(u_mech, T), epsilon(v_mech)) * dx \
                + inner(k_spring * u_mech, v_mech) * ds(1) \
                + inner(k_spring * u_mech, v_mech) * ds(2)

a_mech_robin = lhs(mech_form_robin)
L_mech_robin = rhs(mech_form_robin)

# Setup Solver
mech_prob_robin = LinearVariationalProblem(a_mech_robin, L_mech_robin, u, constant_jacobian=True)
mech_solver_robin = LinearVariationalSolver(mech_prob_robin)

In [ ]:
mech_solver_robin.solve()

s = sigma(u, T)
s_xx, s_yy = s[0,0], s[1,1]
s_xy = s[0,1]
von_mises_s.project(sqrt(s_xx**2 + s_yy**2 - s_xx*s_yy + 3*s_xy**2)/1e6)

fig, ax = plt.subplots(figsize=(10, 4))

cont = tricontourf(von_mises_s, axes=ax, levels=100, cmap='viridis')
fig.colorbar(cont, ax=ax, label='von Mises Stress [MPa]')
ax.set_aspect('equal')
ax.set_title("Thermal Stress (Elastic Support)")

plt.tight_layout()
plt.show()

In [ ]:
scale_factor = 100.0

fig, ax = plt.subplots(figsize=(10, 3))

u_mag = Function(V, name="Displacement Magnitude")
u_mag.interpolate(sqrt(u[0]**2 + u[1]**2))

orig_coords = mesh.coordinates.dat.data.copy()

mesh.coordinates.dat.data[:] += u.dat.data * scale_factor

cont = tricontourf(u_mag, axes=ax, levels=100)
fig.colorbar(cont, ax=ax, label='Displacement Magnitude [m]')

ax.set_title(f"Deformed Shape Colored by Displacement (scaled {scale_factor}x)")
ax.set_aspect('equal')
ax.axis('off')

plt.tight_layout()
plt.show()

mesh.coordinates.dat.data[:] = orig_coords

In this case, the solver finds the equilibrium position where the internal thermal expansion is blanced by the elastic resistance of the boundary. Since the temperature is graded through the thickness, the thermal expansion is larger on the exposed side, leading to a warping that was not captured in either of the previous test cases. The flexibility of the support must be accounted for.